# IEEE-CIS — reconstructed random-vs-time validation audit

**Status: NOT EXECUTED.** This is a **new, from-scratch reconstruction**, not the missing original Kaggle notebook and not evidence for the historical 0.971 / 0.921 numbers.

Requires licensed Kaggle IEEE-CIS competition file `train_transaction.csv`. Set `IEEE_CIS_DATA_DIR` to the directory containing it. Never commit the dataset or Kaggle credentials.

The notebook computes **new** random 80/20 and chronological 80/20 results with identical simple features/model. It also quantifies a **card/address/email proxy overlap**, not verified customer identity. The resulting metrics may differ substantially from historical notebook scores. No output is included until a genuine run is completed.

In [ ]:
from pathlib import Path
import hashlib, json, os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

SEED = 42
DATA_DIR = Path(os.environ.get("IEEE_CIS_DATA_DIR", "data/ieee-cis"))
SOURCE = DATA_DIR / "train_transaction.csv"
if not SOURCE.is_file():
    raise FileNotFoundError(f"Licensed competition data not found at {SOURCE}. Obtain IEEE-CIS access through Kaggle; never invent rows.")
sha256 = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
data = pd.read_csv(SOURCE, low_memory=False)
assert {"TransactionDT", "isFraud", "TransactionID"} <= set(data.columns)
data = data.sort_values(["TransactionDT", "TransactionID"], kind="stable").reset_index(drop=True)
print("Loaded rows:", len(data), "SHA256:", sha256)

In [ ]:
numeric = [c for c in ["TransactionAmt", "card1", "card2", "card3", "card5", "card6", "addr1", "dist1", "C1", "D1"] if c in data.columns]
categorical = [c for c in ["ProductCD", "card4", "P_emaildomain"] if c in data.columns]
assert numeric and categorical, "Expected competition feature columns missing"
features = data[numeric + categorical].copy()
for col in numeric:
    features[col] = pd.to_numeric(features[col], errors="coerce")
for col in categorical:
    features[col] = features[col].astype("string").fillna("__MISSING__").astype(str)
labels = data["isFraud"].astype(int).to_numpy()
time_cut = int(len(data) * .8)
splits = {
    "chronological_80_20": (np.arange(time_cut), np.arange(time_cut, len(data))),
    "stratified_random_80_20": train_test_split(np.arange(len(data)), test_size=.2, random_state=SEED, stratify=labels),
}
print("Held-out row counts:", {k: len(v[1]) for k,v in splits.items()})

In [ ]:
def make_pipeline():
    return Pipeline([
        ("prepare", ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), numeric),
            ("cat", Pipeline([
                ("fill", SimpleImputer(strategy="most_frequent")),
                ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
            ]), categorical),
        ], sparse_threshold=0)),
        ("classifier", HistGradientBoostingClassifier(max_iter=110, max_depth=6, random_state=SEED)),
    ])

# This is a *proxy* group, NOT an authenticated customer identifier.
proxy_cols = [c for c in ["card1", "addr1", "P_emaildomain"] if c in data.columns]
proxy = data[proxy_cols].fillna("<missing>").astype(str).agg("|".join, axis=1).to_numpy()
results = {}
for split_name, (fit_idx, test_idx) in splits.items():
    model = make_pipeline()
    model.fit(features.iloc[fit_idx], labels[fit_idx])
    predicted = model.predict_proba(features.iloc[test_idx])[:, 1]
    fit_proxies = set(proxy[fit_idx])
    results[split_name] = {
        "roc_auc": float(roc_auc_score(labels[test_idx], predicted)),
        "average_precision": float(average_precision_score(labels[test_idx], predicted)),
        "test_count": int(len(test_idx)),
        "test_fraud_rate": float(labels[test_idx].mean()),
        "validation_proxy_seen_in_train_fraction": float(np.mean([x in fit_proxies for x in proxy[test_idx]])),
    }
print(json.dumps(results, indent=2))

In [ ]:
record = {
    "status": "measured_from_provided_local_source",
    "method": "new fixed-feature HistGradientBoosting random-vs-time 80/20 audit, not original Kaggle study",
    "source_file": SOURCE.name,
    "source_sha256": sha256,
    "source_rows": int(len(data)),
    "seed": SEED,
    "features": numeric + categorical,
    "proxy_cols": proxy_cols,
    "results": results,
    "roc_auc_random_minus_time": results["stratified_random_80_20"]["roc_auc"] - results["chronological_80_20"]["roc_auc"],
}
out = Path(os.environ.get("IEEE_CIS_AUDIT_OUTPUT", "research/outputs/leakage_reconstruction.json"))
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(record, indent=2), encoding="utf-8")
print("Wrote measured output:", out, "(review before publication)")